# Passo 2: Transformação (Landing para Bronze - Delta Lake)
Lemos o CSV da `landing-zone` e gravamos como Delta Lake na `bronze`.

In [1]:
from pyspark.sql import SparkSession

# Inicializa o Spark com pacotes do Delta Lake e MinIO
spark = SparkSession.builder \
    .appName("Landing-To-Bronze") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Lendo o CSV da landing-zone
landing_path = "s3a://landing-zone/eventos_csv"
df_csv = spark.read.csv(landing_path, header=True, inferSchema=True)
df_csv.show()

# Escrevendo em formato Delta na bronze
bronze_path = "s3a://bronze/eventos_delta"
df_csv.write.format("delta").mode("overwrite").save(bronze_path)

print("Tabela Delta criada com sucesso na Bronze!")
